In [12]:
import os
import datetime
import pandas as pd
from typing import List, Dict, Any
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter, TokenTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader, PyPDFLoader, PyMuPDFLoader, Docx2txtLoader, UnstructuredWordDocumentLoader

print("Set up completed!")

Set up completed!


### Document Structure In Langchain

In [6]:

doc = Document(
    page_content="This is the main text content",
    metadata={
        "source": "example.txt",
        "page": 1,
        "author": "Ido madaRR",
        "date_created": datetime.datetime.now(),
    },
)

print(f"Document Structure: {doc}")
print("Metadata is crucial for: \n*Filtering search results\n*Tracking document sources\n*Provide context in responses.")

Document Structure: page_content='This is the main text content' metadata={'source': 'example.txt', 'page': 1, 'author': 'Ido madaRR', 'date_created': datetime.datetime(2026, 4, 30, 21, 58, 2, 834067)}
Metadata is crucial for: 
*Filtering search results
*Tracking document sources
*Provide context in responses.


### Text Files (.txt)

In [45]:
os.makedirs("data/text_files", exist_ok=True)

sample_texts = {
    "data/text_files/python_intro.txt": """
    Python is a high-level programming language known for its simplicity and readability. It is widely used by beginners and professionals alike because its syntax is clean and easy to understand, almost like reading English.

Python is a versatile language that can be used for many purposes, including web development, automation, data analysis, artificial intelligence, and more. One of its biggest advantages is the huge ecosystem of libraries and frameworks, which allow developers to build powerful applications without starting from scratch.

For example, with just a few lines of code, you can create scripts to automate repetitive tasks, build APIs, or analyze large datasets. Python also has a strong community, which means there are plenty of tutorials, tools, and support available online.
    """,
    "data/text_files/machine_learning.txt": """
    Machine Learning is a field within computer science that focuses on enabling computers to learn from data instead of being explicitly programmed. Instead of writing rules for every situation, we train models to recognize patterns and make decisions based on examples.

For instance, a machine learning model can be trained to recognize images, predict future trends, or understand human language. The more data the model is exposed to, the better it becomes at making accurate predictions.

There are different types of machine learning, such as:

Supervised learning – learning from labeled data (e.g., spam detection)
Unsupervised learning – finding patterns in unlabeled data (e.g., grouping users)
Reinforcement learning – learning through trial and error (e.g., game-playing AI)

Machine learning is used in many real-world applications like recommendation systems (Netflix, Spotify), voice assistants, fraud detection, and even self-driving cars.
    """,
}

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample texts files created")


Sample texts files created


### TextLoader - Read Single File

In [46]:
loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()

print(f"Loaded {len(document)} document")
print(f"Content: {document[0].page_content[:50]}...")
print(f"Metadata: {document[0].metadata}")

Loaded 1 document
Content: 
    Python is a high-level programming language k...
Metadata: {'source': 'data/text_files/python_intro.txt'}


### TextLoader - Read Multiple Files

In [47]:
dir_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt",           # Pattern to match files
    loader_cls=TextLoader,                  # Loader class
    loader_kwargs={'encoding': 'utf-8'},    # formatting
    show_progress=True
)

documents = dir_loader.load()
print(f"Loaded {len(documents)} documents")
print(documents)

100%|██████████| 2/2 [00:00<00:00, 3070.50it/s]

Loaded 2 documents
[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='\n    Machine Learning is a field within computer science that focuses on enabling computers to learn from data instead of being explicitly programmed. Instead of writing rules for every situation, we train models to recognize patterns and make decisions based on examples.\n\nFor instance, a machine learning model can be trained to recognize images, predict future trends, or understand human language. The more data the model is exposed to, the better it becomes at making accurate predictions.\n\nThere are different types of machine learning, such as:\n\nSupervised learning – learning from labeled data (e.g., spam detection)\nUnsupervised learning – finding patterns in unlabeled data (e.g., grouping users)\nReinforcement learning – learning through trial and error (e.g., game-playing AI)\n\nMachine learning is used in many real-world applications like recommendation systems (Netflix

### Text Splitting Strategies

In [60]:
### Method 1 - Character Splitter
# NOTICE: CharacterTextSplitter respects the paragraph boundary over the size limit — it won't cut mid-sentence, that why we can get warning like:
# "Created a chunk of size 220, which is longer than the specified 200"

char_splitter = CharacterTextSplitter(
    separator="\n",         # The character(s) to split on.
    chunk_size=200,         # Maximum size of each chunk. A chunk won't exceed this limit.
    chunk_overlap=20,       # How many characters are shared between the end of one chunk and the start of the next (10–20% of chunk siz).
    length_function=len     # How chunk size is measured
)
text = documents[0].page_content
char_chunks = char_splitter.split_text(text)

print(len(char_chunks))
print(char_chunks[0])
print("-------------------")
print(char_chunks[1])
print("-------------------")
print(char_chunks[2])

Created a chunk of size 271, which is longer than the specified 200
Created a chunk of size 220, which is longer than the specified 200


5
Machine Learning is a field within computer science that focuses on enabling computers to learn from data instead of being explicitly programmed. Instead of writing rules for every situation, we train models to recognize patterns and make decisions based on examples.
-------------------
For instance, a machine learning model can be trained to recognize images, predict future trends, or understand human language. The more data the model is exposed to, the better it becomes at making accurate predictions.
-------------------
There are different types of machine learning, such as:
Supervised learning – learning from labeled data (e.g., spam detection)


In [64]:
### Method 2 - Recursive Character Splitter (RECOMMENDED)

recursive_splitter = RecursiveCharacterTextSplitter(
    # separators=["\n\n", "\n", " ", ""],
    separators=["\n"],
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

text = documents[0].page_content
char_chunks = recursive_splitter.split_text(text)

print(len(char_chunks))
print(char_chunks[0])
print("-------------------")
print(char_chunks[1])
print("-------------------")
print(char_chunks[2])
print("-------------------")
print(char_chunks[3])

5

    Machine Learning is a field within computer science that focuses on enabling computers to learn from data instead of being explicitly programmed. Instead of writing rules for every situation, we train models to recognize patterns and make decisions based on examples.
-------------------

For instance, a machine learning model can be trained to recognize images, predict future trends, or understand human language. The more data the model is exposed to, the better it becomes at making accurate predictions.
-------------------
There are different types of machine learning, such as:

Supervised learning – learning from labeled data (e.g., spam detection)
-------------------
Unsupervised learning – finding patterns in unlabeled data (e.g., grouping users)
Reinforcement learning – learning through trial and error (e.g., game-playing AI)


In [65]:
### Method 3 - Token-based Splitter (Recommend only when working with token-limited models - a bit slow)
token_splitter = TokenTextSplitter(
    chunk_size=50,
    chunk_overlap=10,
)

token_chunks = token_splitter.split_text(text)
print(len(char_chunks))
print(char_chunks[0])
print("-------------------")
print(char_chunks[1])
print("-------------------")
print(char_chunks[2])
print("-------------------")
print(char_chunks[3])

5

    Machine Learning is a field within computer science that focuses on enabling computers to learn from data instead of being explicitly programmed. Instead of writing rules for every situation, we train models to recognize patterns and make decisions based on examples.
-------------------

For instance, a machine learning model can be trained to recognize images, predict future trends, or understand human language. The more data the model is exposed to, the better it becomes at making accurate predictions.
-------------------
There are different types of machine learning, such as:

Supervised learning – learning from labeled data (e.g., spam detection)
-------------------
Unsupervised learning – finding patterns in unlabeled data (e.g., grouping users)
Reinforcement learning – learning through trial and error (e.g., game-playing AI)


### PDF Files (.pdf)

In [6]:
# Method 1 - PyPDFLoader (Good for most PDFs)
try:
    pypdf_loader = PyPDFLoader("data/pdf/terms_and_condition.pdf")
    pypdf_doc = pypdf_loader.load()
    print(pypdf_doc)
except Exception as e:
    print(f"Error loading PyPDF: {e}")

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)


[Document(metadata={'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Pages', 'creationdate': "D:20260425044100Z00'00'", 'title': '\u200e\u2068נוסח סופי - תנאי שימוש אפליקציה נהג צעיר 25.12.2025\u2069', 'moddate': "D:20260425044100Z00'00'", 'source': 'data/pdf/terms_and_condition.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}, page_content='צעיר הפניקס אפליקציית שימוש תנאי\n :כללי\n.1 ״הפניקס הביטוח תכנית של הרכב באפליקציית שימוש לעשות שבחרת שמחים אנו, יקר מבוטח\n ,הביטוח את ולתפעל לרכוש ניתן דרכה אשר"( החברה"/"הפניקס, "״האפליקציה״: להלן )צעיר״\n\t .הפניקס של,( הפוליסה במסמכי לצפות\n.2 או בשירות שימוש, האפליקציה באמצעות כלשהי פעולה תבצע ובטרם לאפליקציה כניסתך עם\n ומדיניות השימוש תנאי את ולאשר בעיון לקרוא מתבקש הנך, באפליקציה הקיים כלשהו במידע\n או/ו בה הכלול כל על, זו באפליקציה השימוש, ליבך לתשומת. להלן המפורטים הפרטיות\n באפליקציה השימוש ועצם להלן המפורטים לתנאים כפוף, לעת מעת שישתנה וכפי באמצעותה\n התחייבות וכן החברה ידי על בהם שיבוצע שינוי ולכל ל

In [10]:
# Method 2 - PyMuPDFLoader (Good for extraction data inside pdf like images)
try:
    pymupdf_loader = PyMuPDFLoader("data/pdf/sample_explain.pdf")
    pypdf_doc = pymupdf_loader.load()
    print(pypdf_doc)
except Exception as e:
    print(f"Error loading PyPDF: {e}")

[Document(metadata={'producer': 'Acrobat Distiller 4.05 for Windows', 'creator': '', 'creationdate': 'D:20011026133934', 'source': 'data/pdf/sample_explain.pdf', 'file_path': 'data/pdf/sample_explain.pdf', 'total_pages': 4, 'format': 'PDF 1.3', 'title': 'PDF Bookmark Sample', 'author': 'Accelio Corporation', 'subject': '', 'keywords': '', 'moddate': '2001-10-26T13:40:41-04:00', 'trapped': '', 'encryption': 'Standard V1 R2 40-bit RC4', 'modDate': "D:20011026134041-04'00'", 'creationDate': 'D:20011026133934', 'page': 0}, page_content='PDF Bookmark Sample \nPage 1 of 4 \n \nPDF BOOKMARK SAMPLE \n \nSample Date: \nMay 2001 \nPrepared by: \nAccelio Present Applied Technology \nCreated and Tested Using: \n• \nAccelio Present Central 5.4 \n• \nAccelio Present Output Designer 5.4 \nFeatures Demonstrated: \n• \nPrimary bookmarks in a PDF file. \n• \nSecondary bookmarks in a PDF file. \nOverview \nThis sample consists of a simple form containing four distinct fields. The data file contains eight

### Handling PDF Challenges

In [15]:
class SmartPDFCleaner:
    """" Advanced PDF processing """

    def __init__(self, chunk_size=1000, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=[" "])

    def _clean_text(self, page_content):
        # Remove whitespaces
        cleaned_text = " ".join(page_content.split())
        return cleaned_text

    def process_pdf(self, pdf_path: str) -> List[Document]:
        # Load PDF
        pdf_loader = PyPDFLoader(pdf_path)
        pages = pdf_loader.load()

        # Process each page
        processed_chunks = []
        for page_num, page in enumerate(pages):
            # Cleaning
            cleaned_text = self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_cleaner",
                    "char_count": len(cleaned_text),
                }]
            )

            processed_chunks.extend(chunks)

        return processed_chunks




In [20]:
cleaner = SmartPDFCleaner()

## Process a PDF if available
try:
    smart_chunks = cleaner.process_pdf("data/pdf/terms_and_condition.pdf")
    print(f"Processed into {len(smart_chunks)} smart_chunks")
    # print(smart_chunks)

    if smart_chunks:
        print("Sample chunk metadata:")
        for key, value in smart_chunks[0].metadata.items():
            print(f"   {key}: {value}")

except Exception as e:
    print(f"Error processing {e}")

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)


Processed into 35 smart_chunks
Sample chunk metadata:
   producer: macOS Version 26.2 (Build 25C56) Quartz PDFContext
   creator: Pages
   creationdate: D:20260425044100Z00'00'
   title: ‎⁨נוסח סופי - תנאי שימוש אפליקציה נהג צעיר 25.12.2025⁩
   moddate: D:20260425044100Z00'00'
   source: data/pdf/terms_and_condition.pdf
   total_pages: 11
   page: 1
   page_label: 1
   chunk_method: smart_pdf_cleaner
   char_count: 2403


### Word Files (.doc, .docx)

In [17]:
### Method 1 - Docx2txtLoader
# NOTICE: x.doc is the old format of word file, so it better to re-save the file with the new format - x.docx

try:
    docx_loader = Docx2txtLoader("data/docx/will.docx")
    docx = docx_loader.load()

    print(f"Processed into {len(docx)} documents")
    print(f"Content {docx[0].page_content[:50]}...")
    print(f"Metadata {docx[0].metadata}")
except Exception as e:
    print(f"Error processing {e}")

Processed into 1 documents
Content 1



צוואה



הואיל: ואין אדם יודע יום פקודתו;

וה...
Metadata {'source': 'data/docx/will.docx'}


In [16]:
### Method 2 - UnstructuredWordDocumentLoader
# NOTICE: with UnstructuredWordDocumentLoader we are getting chunks of data
try:
    unstructured_loader = UnstructuredWordDocumentLoader("data/docx/will.docx", mode="elements")
    unstructured_docx = unstructured_loader.load()

    print(f"Processed into {len(unstructured_docx)} documents")
    print(unstructured_docx[5])
except Exception as e:
    print(f"Error processing {e}")

Processed into 27 documents
page_content='אני מבטלת בזאת כל צוואה אחרת או תוספת לצוואה שנעשתה על ידי בעבר, ומורה כי כל צוואה ו/או תוספת שעשיתי אי פעם, בטלה ומבוטלת, וחסרת כל תוקף. כמו כן, אני שומרת לעצמי את הזכות לשנות, לתקן, להוסיף, לגרוע ואף לבטל את תנאיה והוראות צוואה זו, או חלק ממנה בכל עת שאחפוץ וככל שיעלה הרצון מלפני, ואפילו לא בפני העדים לצוואתי זו, או מי מהם, אלא שכל עוד לא אשנה או אבטל את צוואתי זו, היא תישאר תקפה.' metadata={'source': 'data/docx/will.docx', 'category_depth': 0, 'file_directory': 'data/docx', 'filename': 'will.docx', 'last_modified': '2026-04-30T22:00:40', 'page_number': 1, 'languages': ['heb'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'ListItem', 'element_id': 'aecda7e7b62c4302de1172b954876a49'}


### CSV & Excel Files